# Dev33: Apply 057 SVM Model to 028 Crop Regions

**Goal**: Use the trained SVM from dev20 (trained on 057) and apply it to 028 crop regions (inverse of dev30).

**Workflow**:
1. Load saved SVM model from dev20 (057-trained, 3 classes)
2. Load and preprocess 028 transect (same preprocessing as dev20)
3. Classify the 4 crop regions from dev30
4. Visualize with plot_classification_as_image

**Key difference from dev30**:
- Dev30: 5-class SVM (sediment, rust, dark_bomb, dark_pit, halo) → classify 028 crops
- Dev33: 3-class SVM (training_dark, training_sediment, training_bombs) → classify 028 crops

## Setup and Import

In [ ]:
import importlib
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle
import importlib
import sys
from pathlib import Path

# Add mjosa_code root to path
mjosa_code_root = Path.cwd().parent  # Up to mjosa_code root
sys.path.insert(0, str(mjosa_code_root))

# Import from mjosa_code (NEW structure)
from utils.uhi import georef
from utils.common import config

importlib.reload(georef)
from utils.uhi.georef import *

## Step 1: Load Trained SVM Model from 6_svm_general
**Note**: First run the save cell in dev20 to create the model file!

In [ ]:
# Load the saved SVM model and preprocessing info
model_file = "./saved_data/svm_model_057_no_unknown.pkl"

if not os.path.exists(model_file):
    print("❌ ERROR: Model file not found!")
    print(f"   Looking for: {model_file}")
    print("\n📝 TO CREATE THE MODEL FILE:")
    print("   1. Open dev20 notebook")
    print("   2. Run all cells up to 'Train SVM'")
    print("   3. Run the '💾 SAVE SVM MODEL' cell")
    print("   4. Then come back here!")
    raise FileNotFoundError(f"Model file not found: {model_file}")
else:
    with open(model_file, "rb") as f:
        saved_model_data = pickle.load(f)

    # Extract model and parameters
    trained_model = saved_model_data["model"]
    class_names = saved_model_data["class_names"]
    label_encoder = saved_model_data["label_encoder"]
    preprocessing_params = saved_model_data["preprocessing"]
    best_params = saved_model_data["best_params"]

    print("✅ SVM model loaded successfully!")
    print(f"\n📊 Model Info:")
    print(f"   Classes: {class_names}")
    print(f"   Best C: {best_params['C']}")
    print(f"   Best gamma: {best_params['gamma']}")
    print(f"\n🔬 Preprocessing (will apply to 028):")
    print(f"   Wavelength range: {preprocessing_params['wavelength_range']} nm")
    print(f"   Smoothing: {preprocessing_params['smoothing_method']}")
    print(f"   Normalization: {preprocessing_params['normalization']}")

## Step 2: Load 028 Transect (All 5 Files)

Same as dev30 - we load all 5 files to cover the crop regions.

In [ ]:
# Load 028 transect
transect = load_transect(config.TRANSECT_028_OUTPUT)

transect.list_files()

# Select all 5 files from transect 028
cube = transect.select_files(
    [
        "rad_uhi_20241029_125028_1",
        "rad_uhi_20241029_125028_2",
        "rad_uhi_20241029_125028_3",
        "rad_uhi_20241029_125028_4",
        "rad_uhi_20241029_125028_5",
    ]
)
cube.describe()

In [ ]:
# Apply illumination correction
cube.apply_illumination_correction_v2()
print("✅ Illumination correction applied")

## Step 3: Apply Same Preprocessing as dev30

In [ ]:
# Apply spectral smoothing (same as dev20) - only if smoothing was used
if preprocessing_params["smoothing_method"] is not None:
    cube.apply_spectral_smoothing(
        method=preprocessing_params["smoothing_method"],
        gaussian_sigma=preprocessing_params["smoothing_sigma"],
    )
    print(
        f"✅ Spectral smoothing applied: {preprocessing_params['smoothing_method']} (σ={preprocessing_params['smoothing_sigma']})"
    )
else:
    print("✅ No spectral smoothing ")

In [ ]:
# Crop wavelengths (same as dev30)
wl_min, wl_max = preprocessing_params["wavelength_range"]
cube.apply_wavelength_filter(wavelength_range=(wl_min, wl_max))

print(f"✅ Wavelength cropping complete")
print(f"   Cropped shape: {cube.data_corrected.shape}")
print(f"   Wavelength range: {cube.wavelengths[0]:.1f} - {cube.wavelengths[-1]:.1f} nm")

In [ ]:
# Check if normalization was used in dev20
if preprocessing_params["normalization"] is not None:
    print(f"✅ Normalization: {preprocessing_params['normalization']}")
    # Add normalization code here if needed
else:
    print("✅ No normalization applied (matching dev20 preprocessing)")

## Step 4: Define Crop Regions (Same as dev30)

## Step 5: Set the Loaded Model on the Cube

The `classify_segment()` method uses the model stored in the cube object. We need to set it manually.

In [ ]:
# Set the loaded model on the cube object
# (classify_segment() expects these attributes to exist)
cube.svm_model = trained_model
cube.svm_label_encoder = label_encoder
cube.svm_class_names = class_names

print("✅ Model set on cube object")
print(f"   cube.svm_model: {type(cube.svm_model)}")
print(f"   cube.svm_label_encoder: {type(cube.svm_label_encoder)}")
print(f"   cube.svm_class_names: {cube.svm_class_names}")

In [ ]:
# Define crop regions (UPDATED coordinates from user)
crop_regions = [
    {
        "name": "Crop 1 (Bomb 2 - Rust)",
        "track": 1258,
        "slit": 250,  # UPDATED from 212
        "width": 500,  # UPDATED from 400
        "aspect_ratio": 4,  # UPDATED from 3.5
    },
    {
        "name": "Crop 2 (Bomb 3)",
        "track": 5592,
        "slit": 700,  # UPDATED from 765
        "width": 500,
        "aspect_ratio": 4,  # UPDATED from 3.5
    },
    {
        "name": "Crop 3 (Bomb 1)",
        "track": 5160,
        "slit": 613,
        "width": 500,  # UPDATED from 400
        "aspect_ratio": 4,  # UPDATED from 3.5
    },
    {
        "name": "Crop 4 (Dark pits)",
        "track": 717,
        "slit": 385,
        "width": 500,  # UPDATED from 400
        "aspect_ratio": 4,  # UPDATED from 3.5
    },
]

print(f"📍 Defined {len(crop_regions)} crop regions (UPDATED):")
for i, crop in enumerate(crop_regions, 1):
    print(
        f"   {i}. {crop['name']}: track={crop['track']}, slit={crop['slit']}, width={crop['width']}"
    )

In [ ]:
# Helper function to classify a single crop region (same as dev30)
def test_classify_crop_region_dev33(cube, crop, confidence_threshold=0.5):
    """Classify a cropped region and return results."""
    # Calculate crop bounds
    half_width_slit = crop["width"] // 2
    half_width_track = int(crop["width"] / crop["aspect_ratio"] / 2)

    crop_track_min = max(0, crop["track"] - half_width_track)
    crop_track_max = min(cube.data_corrected.shape[0], crop["track"] + half_width_track)
    crop_slit_min = max(0, crop["slit"] - half_width_slit)
    crop_slit_max = min(cube.data_corrected.shape[1], crop["slit"] + half_width_slit)

    print(f"\n🔍 Classifying: {crop['name']}")
    print(
        f"   Crop bounds: track [{crop_track_min}:{crop_track_max}], slit [{crop_slit_min}:{crop_slit_max}]"
    )

    # Classify the segment
    classification_results = cube.classify_segment(
        segment_start=crop_track_min,
        segment_end=crop_track_max - 1,
        use_corrected=True,
        confidence_threshold=confidence_threshold,
        quiet=False,
    )

    # Print pixel counts
    print(f"\n📊 Pixel counts for {crop['name']}:")
    for class_name in classification_results["class_names"]:
        count = np.sum(classification_results["classification_map"] == class_name)
        percentage = 100.0 * count / classification_results["classification_map"].size
        print(f"   {class_name}: {count} pixels ({percentage:.1f}%)")

    return classification_results, crop_track_min, crop_track_max


# Classify all 4 crop regions
crop_results = []

for crop in crop_regions:
    results, track_min, track_max = test_classify_crop_region_dev33(
        cube, crop, confidence_threshold=0
    )
    crop_results.append(
        {
            "crop": crop,
            "results": results,
            "track_min": track_min,
            "track_max": track_max,
        }
    )

print(f"\n✅ All {len(crop_results)} crop regions classified!")

## Step 6: Classify Each Crop Region

Use the 057-trained model to classify all 4 crop regions separately.

In [ ]:
%matplotlib inline

# Plot each crop region separately with plot_rgb (same as dev30)
print("📊 Plotting classification results for each crop region...")

for crop_result in crop_results:
    crop = crop_result["crop"]
    results = crop_result["results"]
    track_min = crop_result["track_min"]

    # Create ROI collection from classification map
    classification_rois = {}
    for class_name in results["class_names"]:
        mask = results["classification_map"] == class_name
        rows, cols = np.where(mask)
        # Convert to global coordinates (add track_min offset)
        pixels = [(col, row + track_min) for row, col in zip(rows, cols)]
        if len(pixels) > 0:
            classification_rois[class_name] = pixels

    print(f"\n📍 {crop['name']}")
    print(f"   ROIs: {list(classification_rois.keys())}")
    for class_name in sorted(classification_rois.keys()):
        print(f"      {class_name}: {len(classification_rois[class_name])} pixels")

    # Plot with plot_rgb (same visualization as dev30)
    cube.plot_rgb(
        use_corrected=True,
        flip_axes=True,
        flip_horizontal=True,
        figsize=(30, 8.32),
        crop_center_track=crop["track"],
        crop_center_slit=crop["slit"],
        crop_width=crop["width"],
        crop_aspect_ratio=crop["aspect_ratio"],
        display_aspect_ratio=4.0,
        show_file_boundaries=False,
        roi_collection=classification_rois,
        roi_legend_loc="outside",
        roi_marker_size=2,
        roi_legend_markersize=50,
        roi_marker_edgewidth=0,
        roi_overlay_mode="solid",  # ← Enable solid mode
        roi_solid_pixel_size=1,  
    )

## Summary

**What we did:**
1. ✅ Loaded SVM model trained on 057 transect (dev20, 3 classes)
2. ✅ Applied same preprocessing to 028 transect
3. ✅ Defined 4 crop regions (same as dev30)
4. ✅ Classified each crop region separately using 057's model
5. ✅ Visualized each crop with plot_rgb

**Key insight**: Testing if the 057-trained model (3 classes) generalizes to 028 crop regions!

In [ ]:
%matplotlib inline

def plot_classification_as_image(
    cube,
    crop,
    classification_map,
    class_names,
    figsize=(30, 8.32),
    flip_axes=True,
    flip_horizontal=True,
):
    """
    Plot classification results as a proper classified image (not ROI markers).
    Each pixel gets the color of its class - like a normal image.
    """
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np

    # Define colors for each class - FIXED to match actual class names from classify_segment
    class_colors = {
        # Dev33 classes (from classify_segment output - prefixed with "classified_")
        "classified_dark": [0.1, 0.1, 0.1],  # Dark gray/black
        "classified_sediment": [0.6, 0.4, 0.2],  # Brown
        "classified_bombs": [0.8, 0.2, 0.1],  # Red-orange
        "classified_unknown": [0.5, 0.5, 0.5],  # Medium gray
        # Original training names (just in case)
        "training_dark": [0.1, 0.1, 0.1],  # Dark gray/black
        "training_sediment": [0.6, 0.4, 0.2],  # Brown
        "training_bombs": [0.8, 0.2, 0.1],  # Red-orange
        # Dev30 classes
        "sediment": [0.6, 0.4, 0.2],  # Brown
        "rust": [0.8, 0.2, 0.1],  # Red-orange
        "dark_bomb": [0.1, 0.1, 0.1],  # Dark gray/black
        "dark_pit": [0.2, 0.2, 0.2],  # Gray
        "halo": [0.9, 0.9, 0.5],  # Yellow
        "uncertain": [0.5, 0.5, 0.5],  # Medium gray
    }

    # Create RGB image from classification map
    height, width = classification_map.shape
    rgb_image = np.zeros((height, width, 3))

    for class_name in class_names:
        mask = classification_map == class_name
        color = class_colors.get(class_name, [0.5, 0.5, 0.5])  # Default gray
        rgb_image[mask] = color

    # Apply flipping to match plot_rgb behavior
    # Step 1: Transpose to match plot_rgb's initial .T.copy()
    rgb_image = rgb_image.transpose(1, 0, 2)  # (slits, tracks, 3)

    # Step 2: Apply flip_axes (like plot_rgb does)
    if flip_axes:
        rgb_image = rgb_image.transpose(1, 0, 2)  # Back to (tracks, slits, 3)

    # Step 3: Apply flip_horizontal (like plot_rgb does)
    if flip_horizontal:
        rgb_image = np.flip(rgb_image, axis=1)  # Flip along axis 1

    # Calculate extent in global coordinates
    half_width_slit = crop["width"] // 2
    half_width_track = int(crop["width"] / crop["aspect_ratio"] / 2)

    slit_min = crop["slit"] - half_width_slit
    slit_max = crop["slit"] + half_width_slit
    track_min = crop["track"] - half_width_track
    track_max = crop["track"] + half_width_track

    if flip_axes:
        # When axes flipped: x=slit, y=track (matches plot_rgb)
        extent = [slit_min, slit_max, track_min, track_max]
        xlabel, ylabel = "Slit Pixel Index", "Track Index"
    else:
        # Normal: x=track, y=slit
        extent = [track_min, track_max, slit_min, slit_max]
        xlabel, ylabel = "Track Index", "Slit Pixel Index"

    # Create figure
    fig, ax = plt.subplots(figsize=figsize)

    # Display the classified image
    im = ax.imshow(
        rgb_image,
        extent=extent,
        origin="lower",
        aspect=crop.get("display_aspect_ratio", 4.0),
        interpolation="nearest",
    )

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(f"Classification: {crop['name']}", fontsize=14, fontweight="bold")

    # Create legend with class colors
    legend_patches = []
    for class_name in class_names:
        count = np.sum(classification_map == class_name)
        color = class_colors.get(class_name, [0.5, 0.5, 0.5])
        label = f"{class_name} ({count} px)"
        legend_patches.append(mpatches.Patch(color=color, label=label))

    ax.legend(
        handles=legend_patches,
        loc="center left",
        bbox_to_anchor=(1, 0.5),
        frameon=True,
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    return fig, ax


# Plot classification results as proper images (not ROI markers)
print("📊 Plotting classification as IMAGES (not ROI markers)...")

for crop_result in crop_results:  # FIXED: Use crop_results instead of multiclass_crop_results
    crop = crop_result["crop"]
    results = crop_result["results"]

    print(f"\n📍 {crop['name']}")
    print(f"   Classes: {results['class_names']}")

    # CRITICAL FIX: Crop the classification_map to the slit range!
    # classify_segment only crops track dimension, not slit dimension
    half_width_slit = crop["width"] // 2
    crop_slit_min = max(0, crop["slit"] - half_width_slit)
    crop_slit_max = min(
        results["classification_map"].shape[1], crop["slit"] + half_width_slit
    )

    # Crop the classification map to the slit range
    cropped_classification_map = results["classification_map"][
        :, crop_slit_min:crop_slit_max
    ]

    print(f"   Original shape: {results['classification_map'].shape}")
    print(
        f"   Cropped to slit [{crop_slit_min}:{crop_slit_max}]: {cropped_classification_map.shape}"
    )
    print(f"   Pixel counts:")
    for class_name in results["class_names"]:
        count = np.sum(cropped_classification_map == class_name)
        percentage = 100.0 * count / cropped_classification_map.size
        print(f"      {class_name}: {count} pixels ({percentage:.1f}%)")

    # Plot as proper classified image with CROPPED data
    plot_classification_as_image(
        cube,
        crop,
        cropped_classification_map,  # Use CROPPED classification map!
        results["class_names"],
        figsize=(30, 7.18),
        flip_axes=True,
        flip_horizontal=True,
    )

---

## 📊 COMPREHENSIVE CLASSIFICATION ANALYSIS

Generate detailed metrics, visualizations, and statistics for cross-transect transfer (057 model → 028 data).

In [ ]:
# Import the comprehensive analysis function
from utils.uhi.classification_analysis import analyze_classification_results

print("=" * 80)
print("📊 CROSS-TRANSECT TRANSFER ANALYSIS: 057 MODEL → 028 DATA")
print("=" * 80)
print(f"Model: SVM trained on transect 057 (3-class)")
print(f"Application: Applied to transect 028 crop regions")
print(f"Purpose: Test if simpler 057 model (3 classes) can generalize to 028 site")
print("=" * 80)

# For notebook 11, we have crop-based classification results
# Analyze each crop if available, or the most recent 'results' dict
if "results" in globals() and isinstance(results, dict):
    cls_map = results.get("classification_map", None)
    class_names_var = results.get("class_names", None)

    if cls_map is not None:
        print("\n⚠️  NOTE: Analyzing the most recent crop classification.")
        print(
            "    To analyze all crops, re-run this cell after each crop classification.\n"
        )

        # Run comprehensive analysis
        metrics_057_to_028 = analyze_classification_results(
            classification_map=cls_map,
            filtered_map=None,  # Typically no filtering for crop regions
            class_names=class_names_var,
            track_start=0,
            figsize=(14, 6),
            show_plots=True,
            save_csv="./saved_data/classification_summary_057_to_028_latest_crop.csv",
        )

        print("\n" + "=" * 80)
        print("📝 CROSS-TRANSFER NARRATIVE SUMMARY")
        print("=" * 80)
        print(f"The 057 model (trained on 3 classes) was applied to 028 crop regions.")
        print(f"Total pixels in this crop: {metrics_057_to_028['total_pixels']:,}")
        print(f"\nClass distribution in 028 crop when using 057 model:")

        for cn in class_names_var:
            count = metrics_057_to_028["counts_before"].get(cn, 0)
            pct = 100.0 * count / metrics_057_to_028["total_pixels"]
            print(f"  • {cn}: {count:,} pixels ({pct:.2f}%)")

        print("\n🔍 KEY INSIGHT:")
        print(
            "   The 057 model uses 3 classes (dark, sediment, bombs) while 028 has 5."
        )
        print("   Compare with notebook 9 (028 model on 028 data) to see if the")
        print("   simpler 3-class model captures the main seafloor features.")
        print("=" * 80)
    else:
        print("❌ ERROR: 'results' dict found but no classification_map inside!")
        print("   Please run a crop classification cell first.")
else:
    print("❌ ERROR: 'results' variable not found!")
    print("   Please run the crop classification cells first.")
    print("   This analysis will work on the most recently classified crop.")